In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
train_df = pd.read_parquet('train_final.parquet')
test_df = pd.read_parquet('test_final.parquet')

In [5]:
train_df.drop('Fwd Header Length.1',inplace=True,axis=1)
test_df.drop('Fwd Header Length.1',inplace=True,axis=1)

In [6]:
for col in train_df.columns:
    col_type = train_df[col].dtype
    if col_type != object:
        c_min = train_df[col].min()
        c_max = train_df[col].max()
        # Downcasting float64 to float32
        if str(col_type).find('float') >= 0 and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
            train_df[col] = train_df[col].astype(np.float32)

        # Downcasting int64 to int32
        elif str(col_type).find('int') >= 0 and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            train_df[col] = train_df[col].astype(np.int32)

In [7]:
for col in test_df.columns:
    col_type = test_df[col].dtype
    if col_type != object:
        c_min = test_df[col].min()
        c_max = test_df[col].max()
        # Downcasting float64 to float32
        if str(col_type).find('float') >= 0 and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
            test_df[col] = test_df[col].astype(np.float32)

        # Downcasting int64 to int32
        elif str(col_type).find('int') >= 0 and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            test_df[col] = test_df[col].astype(np.int32)

In [8]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1975733 entries, 0 to 1975732
Data columns (total 78 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int32  
 1   Flow Duration                int32  
 2   Total Fwd Packets            int32  
 3   Total Backward Packets       int32  
 4   Total Length of Fwd Packets  int32  
 5   Total Length of Bwd Packets  int32  
 6   Fwd Packet Length Max        int32  
 7   Fwd Packet Length Min        int32  
 8   Fwd Packet Length Mean       float32
 9   Fwd Packet Length Std        float32
 10  Bwd Packet Length Max        int32  
 11  Bwd Packet Length Min        int32  
 12  Bwd Packet Length Mean       float32
 13  Bwd Packet Length Std        float32
 14  Flow Bytes/s                 float32
 15  Flow Packets/s               float32
 16  Flow IAT Mean                float32
 17  Flow IAT Std                 float32
 18  Flow IAT Max                 int32  
 19  

In [9]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 580766 entries, 0 to 580765
Data columns (total 78 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Destination Port             580766 non-null  int32  
 1   Flow Duration                580766 non-null  int32  
 2   Total Fwd Packets            580766 non-null  int32  
 3   Total Backward Packets       580766 non-null  int32  
 4   Total Length of Fwd Packets  580766 non-null  int32  
 5   Total Length of Bwd Packets  580766 non-null  int32  
 6   Fwd Packet Length Max        580766 non-null  int32  
 7   Fwd Packet Length Min        580766 non-null  int32  
 8   Fwd Packet Length Mean       580766 non-null  float32
 9   Fwd Packet Length Std        580766 non-null  float32
 10  Bwd Packet Length Max        580766 non-null  int32  
 11  Bwd Packet Length Min        580766 non-null  int32  
 12  Bwd Packet Length Mean       580766 non-null  float32
 13 

In [10]:
train_df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Destination Port,1975733.0,8.712207e+03,1.916429e+04,0.0,53.0,80.0,443.0,65535.0
Flow Duration,1975733.0,1.749025e+07,3.620186e+07,-4.0,210.0,49173.0,5375456.0,119999998.0
Total Fwd Packets,1975733.0,1.083803e+01,8.297929e+02,1.0,2.0,2.0,6.0,219759.0
Total Backward Packets,1975733.0,1.222618e+01,1.101507e+03,0.0,1.0,2.0,5.0,291922.0
Total Length of Fwd Packets,1975733.0,5.793124e+02,1.135443e+04,0.0,18.0,70.0,352.0,12900000.0
...,...,...,...,...,...,...,...,...
Idle Mean,1975733.0,1.014451e+07,2.633459e+07,0.0,0.0,0.0,0.0,120000000.0
Idle Std,1975733.0,3.496024e+05,3.616483e+06,0.0,0.0,0.0,0.0,76900000.0
Idle Max,1975733.0,1.042373e+07,2.677009e+07,0.0,0.0,0.0,0.0,120000000.0
Idle Min,1975733.0,9.849893e+06,2.618639e+07,0.0,0.0,0.0,0.0,120000000.0


In [11]:
train_df['Attack'].value_counts()

Attack
0    1720966
4     189135
3      29999
6      25501
2       7417
7       1718
1        974
5         23
Name: count, dtype: int64

In [12]:
num_unique = train_df.nunique()
one_variable = num_unique[num_unique == 1]
not_one_variable = num_unique[num_unique > 1].index

dropped_cols = one_variable.index
train_df = train_df[not_one_variable]
test_df = test_df[not_one_variable]

print('Dropped columns:')
dropped_cols

Dropped columns:


Index(['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk',
       'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
       'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'],
      dtype='object')

In [13]:
train_df.shape

(1975733, 70)

In [14]:
from sklearn.preprocessing import StandardScaler
import joblib

features_train = train_df.drop('Attack', axis = 1)
attacks_train = train_df['Attack']
features_test = test_df.drop('Attack', axis = 1)
attacks_test = test_df['Attack']

scaler = StandardScaler()

scaled_features_train = scaler.fit_transform(features_train)
scaled_features_test = scaler.transform(features_test)

In [15]:
joblib.dump(scaler,r"scalers\standardscaler.joblib")

['scalers\\standardscaler.joblib']

In [16]:
print(f"Features Shape Train: f{features_train.shape}")
print(f"Attacks Shape Train: f{attacks_train.shape}")
print(f"Features Shape Test: f{features_test.shape}")
print(f"Attacks Shape Test: f{attacks_test.shape}")

Features Shape Train: f(1975733, 69)
Attacks Shape Train: f(1975733,)
Features Shape Test: f(580766, 69)
Attacks Shape Test: f(580766,)


In [17]:
from sklearn.decomposition import IncrementalPCA

size = len(features_train.columns) // 2
# ipca = IncrementalPCA(n_components = size, batch_size = 500)
# for batch in np.array_split(scaled_features_train, len(features_train) // 500):
#     ipca.partial_fit(batch)

# print(f'information retained: {sum(ipca.explained_variance_ratio_):.2%}')

In [20]:
ipca = joblib.load(r"scalers\incrementail_pca_model.joblib")

In [21]:
transformed_features_train = ipca.transform(scaled_features_train)
transformed_features_test = ipca.transform(scaled_features_test)
new_data_train = pd.DataFrame(transformed_features_train, columns = [f'PC{i+1}' for i in range(size)])
new_data_test = pd.DataFrame(transformed_features_test, columns = [f'PC{i+1}' for i in range(size)])
new_data_train['Attack'] = attacks_train.values
new_data_test['Attack'] = attacks_test.values

In [22]:
# joblib.dump(ipca,r"scalers\incrementail_pca_model.joblib")

In [23]:
new_data_train

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,Attack
0,1.208544,0.232614,-2.655180,0.684672,-0.120985,-0.096676,-1.274258,-1.120255,0.329106,-0.184085,...,-0.078263,-0.882229,-0.173365,0.761955,-0.111812,-0.073425,0.059455,-0.003460,-0.180357,4
1,-1.988981,-0.055431,0.204552,-0.686672,0.642098,0.390195,-0.348629,0.197313,-0.150807,-0.007578,...,-0.019601,-0.268841,-0.268398,-0.795899,-0.919414,-0.013304,0.200705,-0.001187,0.020557,4
2,-1.574411,-0.011218,-0.053708,0.302252,0.434444,1.357662,-1.779988,0.091563,0.223065,-0.074836,...,-0.485837,-0.669890,-0.279383,0.623242,-0.092166,-0.269316,-0.089637,-0.002593,0.178976,4
3,-0.198965,0.233473,-1.569855,1.807272,0.173142,-0.438847,-1.290865,-0.357069,0.292775,-0.131580,...,-0.343115,-0.718166,0.195022,0.941419,-0.487479,-0.274378,-0.170297,-0.006779,0.075665,4
4,-1.989184,-0.055439,0.204645,-0.687136,0.643899,0.389367,-0.347476,0.196930,-0.150219,-0.007371,...,-0.019614,-0.268700,-0.269174,-0.795717,-0.919747,-0.013496,0.200660,-0.001196,0.020574,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975728,-1.892970,-0.059463,0.268084,-0.687860,0.625140,0.404421,-0.364730,0.208069,-0.166739,-0.007259,...,-0.034621,-0.262164,-0.260249,-0.786141,-0.883940,-0.032408,0.190146,-0.002098,0.076555,3
1975729,7.703800,0.383432,-8.458488,-3.673742,-2.100768,-0.083736,-0.681627,-5.125103,-0.126348,-0.585330,...,0.484611,-1.316188,-0.451711,-0.533258,0.067314,-0.361592,-0.270713,-0.002198,0.359764,3
1975730,10.372289,0.403159,-3.090784,-0.735611,0.920430,6.009205,6.293182,-4.362832,-2.597992,-0.258228,...,0.472888,-0.046572,-0.497008,-0.665924,-0.034314,-0.263707,0.040527,-0.001518,0.020191,3
1975731,6.447191,-0.218121,-0.766414,-2.369543,0.609421,1.122436,1.216520,-0.676654,-0.521771,0.095917,...,-0.649322,1.134192,0.933602,0.297727,-1.345327,0.405216,0.053539,-0.001052,0.364618,3


In [24]:
new_data_test

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,Attack
0,0.396478,-0.017140,0.736656,1.655310,0.288288,1.395011,-1.628344,0.763167,0.839899,0.003989,...,-0.346538,0.058353,0.194645,-0.366513,0.818286,-0.411664,-0.159089,-0.007298,0.189345,1
1,-2.205064,-0.071471,0.336313,-1.113233,0.983026,0.555444,-0.807218,0.926283,-1.070795,-0.198375,...,-0.010669,0.066165,-1.396396,0.492691,-0.742902,-0.276914,0.049973,-0.014808,0.025603,1
2,-1.606243,0.068772,-0.582775,0.733688,0.295551,0.659243,-1.309632,-0.133764,0.433704,-0.040781,...,-0.428591,-0.729117,0.216192,0.882142,0.249216,-0.095325,0.012558,0.002916,-0.076607,1
3,0.204688,0.501276,-2.798459,5.441145,1.065465,-3.253276,-1.801238,-0.023221,-0.372418,-0.349266,...,-0.139342,-0.167110,-0.342030,0.896239,0.308911,0.254501,0.245364,0.001174,-0.086356,1
4,-2.225264,-0.071056,0.350495,-1.101375,0.898293,0.406281,-0.613577,0.895708,-1.042461,-0.178305,...,0.022753,0.085876,-1.604087,0.496897,-0.908449,-0.251974,0.036758,-0.014039,0.054440,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
580761,9.958669,38.390728,-2.692725,53.710183,15.094097,38.597739,54.580539,-14.295906,-28.731792,-4.339113,...,1.636052,-1.268645,-0.587599,-2.049051,0.065660,-0.333619,-5.280212,0.354326,-3.012593,5
580762,-1.518452,-0.066873,0.488549,-0.254847,0.654798,0.010255,-0.497535,-0.030764,-0.300296,-0.057770,...,-0.020726,-0.150591,-0.222207,-0.735456,-0.855979,-0.119345,0.133783,-0.005866,0.411502,5
580763,3.157464,0.007253,2.502713,2.060814,3.854391,-0.625714,3.242231,1.568658,0.882725,0.388804,...,-0.452920,-0.052739,-0.389703,-0.645854,-0.461773,0.264834,0.101332,0.020591,-0.648312,5
580764,4.362206,-0.020204,3.061672,2.440110,3.864360,-1.123107,2.613242,2.302659,1.092939,0.366931,...,-0.161412,-0.314129,-0.460134,-0.969392,-0.453448,-0.249827,-0.219978,0.007026,-0.344661,5


In [25]:
new_data_train.to_parquet(r'final\new_data_train.parquet')
new_data_test.to_parquet(r'final\new_data_test.parquet')

Balanced Dataset for binary Clasification

In [29]:
new_data_train['Attack'].value_counts()

Attack
0    1720966
4     189135
3      29999
6      25501
2       7417
7       1718
1        974
5         23
Name: count, dtype: int64

In [39]:
normal_traffic_train = new_data_train.loc[new_data_train['Attack'] == 0]
intrusions_train = new_data_train.loc[new_data_train['Attack'] != 0]

In [40]:
# Run these two lines to see the counts:
print(f"Normal Train Count: {len(normal_traffic_train)}")
print(f"Intrusions Train Count: {len(intrusions_train)}")

Normal Train Count: 1720966
Intrusions Train Count: 254767


In [43]:
normal_traffic_train = normal_traffic_train.sample(n = len(intrusions_train), replace = False, random_state=42)
ids_data_train = pd.concat([intrusions_train, normal_traffic_train])
ids_data_train['Attack'] = np.where((ids_data_train['Attack'] == 0), 0, 1)
bc_data_train = ids_data_train.sample(n = 30000, random_state=42)

In [44]:
# 6a. Print class distribution for the final training subset
print("Distribution for Training Subset (bc_data_train):")
print(bc_data_train['Attack'].value_counts())

Distribution for Training Subset (bc_data_train):
Attack
1    15238
0    14762
Name: count, dtype: int64


In [ ]:

normal_traffic_test = new_data_test.loc[new_data_test['Attack'] == 0]
intrusions_test = new_data_test.loc[new_data_test['Attack'] != 0]

print("-" * 30)
normal_traffic_test = normal_traffic_test.sample(n = len(intrusions_test), replace = False, random_state=42)

ids_data_test = pd.concat([intrusions_test, normal_traffic_test])
ids_data_test['Attack'] = np.where((ids_data_test['Attack'] == 0), 0, 1)


bc_data_test = ids_data_test.sample(n=15000,random_state=42) 

print("Distribution for Balanced Test Set (bc_data_test):")
print(bc_data_test['Attack'].value_counts())

Distribution for Training Subset (bc_data_train):
Attack
1    15238
0    14762
Name: count, dtype: int64
------------------------------
Distribution for Balanced Test Set (bc_data_test):
Attack
1    7613
0    7387
Name: count, dtype: int64


In [46]:
bc_data_train.to_parquet(r'final\train_bc.parquet')
bc_data_test.to_parquet(r'final\test_bc.parquet')

In [47]:
bc_data_train

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,Attack
1203104,-2.072673,-0.039933,0.246683,-0.075939,-1.548547,-0.342667,0.577823,0.127821,0.048951,0.049428,...,-0.305897,-0.143233,-0.166687,-0.231089,0.046109,0.262570,-0.132401,0.012278,0.034832,0
92626,9.232512,-0.323045,-0.744081,-2.681784,0.089277,-0.646570,0.546174,-0.427273,-1.070160,-0.057212,...,-0.706232,0.368901,0.411417,-0.172831,-0.301961,-0.019235,-0.017480,-0.001117,0.299217,1
125729,14.085315,-0.623100,1.606880,-1.277680,-0.615191,-0.289676,1.130447,5.030051,1.308825,0.482177,...,0.157119,-0.184401,0.223163,0.079090,-0.383106,0.140866,0.026938,0.003030,0.363849,1
1237720,-1.546070,-0.114485,0.798583,-0.644667,0.709688,0.415843,-0.617887,-0.091776,-0.264529,-0.048773,...,-0.107176,-0.176664,-0.238419,-0.780809,-0.840451,-0.194065,0.213124,-0.009912,0.477741,0
114438,10.777613,-0.335384,-1.080721,-2.951198,0.459364,-0.697217,0.604147,-0.687188,-1.033221,-0.040631,...,-0.277981,0.208135,0.191413,0.345029,-0.109213,0.020250,-0.058897,0.002049,-0.172678,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
583443,-2.008849,-0.038294,0.198538,-0.170766,-1.722766,-0.351770,0.684410,0.067556,0.073057,0.057585,...,0.359408,0.078479,0.026619,-0.009968,0.025371,-0.149635,0.034525,-0.006851,-0.031870,0
81109,16.116166,-0.484559,-0.758974,-2.116525,-0.540423,-0.475223,1.083419,3.869279,1.374924,0.348945,...,1.287712,-1.348303,-1.402717,-0.486299,1.080151,-0.379543,-0.095246,0.003042,-0.682065,1
695876,-2.067602,-0.036555,0.244673,-0.090382,-1.747112,-0.521969,0.812498,0.074217,0.049940,0.063280,...,0.036413,-0.026437,-0.068972,-0.094888,0.056891,0.130282,-0.069027,0.005984,0.014318,0
22719,10.151793,-0.387998,-0.475682,-2.985745,0.502888,-0.408139,0.610489,-0.424123,-0.951794,0.012271,...,-0.364112,0.385644,0.490673,0.532534,-0.442840,0.239525,0.057213,0.001591,-0.067826,1


In [48]:
bc_data_test

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,Attack
554110,8.509515,-1.542712,16.278398,2.403198,-1.799757,-1.176909,-3.016289,-4.374014,-0.536139,-0.529909,...,-0.275772,0.254021,-0.025670,0.003009,-0.003199,0.222446,-0.017218,0.008281,0.027068,0
355136,-1.021278,0.254915,-1.064661,0.883591,-0.256423,0.371075,-1.218398,-0.220051,0.253150,-0.105640,...,-0.797908,-0.555513,-0.094093,0.343454,0.251316,-0.167332,0.118907,-0.001653,-0.177433,0
453576,3.232375,0.311041,-2.115942,1.491144,-0.140791,1.945023,-0.614943,-0.679277,0.644834,-0.120625,...,-0.109200,0.386158,0.408211,-0.235913,-0.204110,0.048353,-0.027453,-0.000754,0.064942,0
37330,-1.939251,0.027809,-0.377410,0.372929,-0.078089,1.246674,-1.472499,-0.175035,0.564796,-0.033089,...,-0.411485,-0.303636,0.060571,-0.042004,0.604346,-0.392453,-0.073986,-0.003118,-0.075244,1
144939,4.151673,0.274390,-5.390591,-2.030859,-1.869279,0.260969,-0.755279,-3.009522,0.002512,-0.383550,...,-0.615807,-0.512703,0.653986,-0.190255,-0.852566,0.039728,-0.016825,-0.000127,0.255223,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355108,-2.151778,-0.033285,0.391855,0.331109,-2.854796,-1.492109,1.718229,-0.132350,-0.041754,0.090483,...,-0.455777,-0.094294,-0.161736,-0.052037,0.212741,0.837146,-0.321045,0.038966,0.166208,0
130273,3.838633,0.257139,-5.086801,-1.887613,-1.827806,0.268312,-0.747304,-2.842166,0.021309,-0.363334,...,-0.652266,-0.490095,0.816058,-0.114465,-1.046994,0.005691,-0.054993,-0.005763,0.418377,1
111190,-1.711587,-0.079166,0.404037,-0.703029,0.640025,0.404309,-0.370914,0.201075,-0.185086,-0.003086,...,-0.063660,-0.248838,-0.264260,-0.761970,-0.826870,-0.077844,0.162382,-0.004133,0.207788,1
572549,-0.055718,-0.322888,3.318438,0.177561,1.356707,0.896121,-3.228475,-3.982850,-0.761935,-0.501722,...,1.216364,0.513353,0.411947,-2.137555,-0.127670,-0.281971,0.029263,-0.009991,-0.037331,0


BAlANCED DATA FOR MULTICLASS CLASSIFICATION

In [58]:
new_data_test['Attack'].value_counts()

Attack
0    395106
3     98022
6     79290
4      4871
2      2039
1       981
7       433
5        24
Name: count, dtype: int64

In [62]:
# 1. Count classes
class_counts_train = new_data_train['Attack'].value_counts()

# 2. Filter out small classes (where count < 900)
selected_classes_train = class_counts_train[class_counts_train >= 900] # Use >= to include 900
class_names_train = selected_classes_train.index

# 3. Filter the main DataFrame to include only selected classes
selected_train = new_data_train[new_data_train['Attack'].isin(class_names_train)] # FIXED variable name

dfs = []
for name in class_names_train:
    df = selected_train[selected_train['Attack'] == name]

    if len(df) > 2500:
        df = df.sample(n = 5000, random_state = 42, replace = False)

    dfs.append(df)

# 5. Combine all processed DataFrames (filtered small classes, subsampled large classes)
mc_train = pd.concat(dfs, ignore_index = True)

# 6. Print the distribution of the final DataFrame
print(mc_train['Attack'].value_counts()) # FIXED variable name

Attack
0    5000
4    5000
3    5000
6    5000
2    5000
7    1718
1     974
Name: count, dtype: int64


In [63]:

MAX_SAMPLES = 2500


class_counts_test = new_data_test['Attack'].value_counts()

# 2. Filter out small classes (where count < 900)
# We keep classes with 900 or more records.
selected_classes_test = class_counts_test[class_counts_test >= 400]
class_names_test = selected_classes_test.index

# 3. Filter the main DataFrame to include only selected classes
selected_test = new_data_test[new_data_test['Attack'].isin(class_names_test)] 

dfs_test = []

for name in class_names_test:
    df_test = selected_test[selected_test['Attack'] == name]

    if len(df_test) > MAX_SAMPLES:
        df_test = df_test.sample(n = MAX_SAMPLES, random_state = 42, replace = False)

    dfs_test.append(df_test)

# 5. Combine all processed DataFrames (filtered small classes, subsampled large classes)
mc_test = pd.concat(dfs_test, ignore_index = True)

# 6. Print the distribution of the final DataFrame
print("\nFinal Multiclass Test Data Distribution:")
print(mc_test['Attack'].value_counts())


Final Multiclass Test Data Distribution:
Attack
0    2500
3    2500
6    2500
4    2500
2    2039
1     981
7     433
Name: count, dtype: int64


In [65]:
mc_train.to_parquet(r'final\train_mc.parquet')
mc_test.to_parquet(r'final\test_mc.parquet')